# SE-100M fine-tune vs from-scratch STATE on PBMC

Companion to `2026-04-21_14-00_compute_finetune_pretrained_state_pbmc_noise_scaling.py`.
Compares (a) a 1-epoch fine-tune of the pretrained SE-100M model against (b) the from-scratch STATE baseline already produced by `2026-04-16_14-49_compute_state_all_datasets.py`.

Both runs share the PBMC 10-size x 10-quality grid and the same MI signals (`celltype.l3`, `protein_counts`).

In [ ]:
# Collect loss curves + MI for BOTH:
#   * fine-tune (SE-100M) under ~/noise_scaling/data/other/finetunning_state/finetune_00/<size>/<quality>/
#   * from-scratch STATE under ~/noise_scaling/data/PBMC/<size>/<quality>/results/State/model/
# Both emit the same Lightning RobustCSVLogger metrics.csv schema, so the readers are shared.
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

FT_ROOT   = Path('/home/igor/noise_scaling/data/other/finetunning_state/finetune_00')
SC_ROOT   = Path('/home/igor/noise_scaling/data/PBMC')
MI_SEED   = 42
SIGNAL_RE = re.compile(r'Y_(.+?)_[0-9][0-9_.]*$')


def _read_metrics(metrics_csv: Path):
    """Return (train_df[step,train_loss], val_df[step,val_loss]) or (None, None)."""
    if not metrics_csv.exists() or metrics_csv.stat().st_size == 0:
        return None, None
    try:
        m = pd.read_csv(metrics_csv, on_bad_lines='skip')
    except Exception:
        return None, None
    tcol, vcol = 'trainer/train_loss', 'validation/val_loss'
    tr = (m.dropna(subset=[tcol])[['step', tcol]]
            .rename(columns={tcol: 'train_loss'}).sort_values('step').reset_index(drop=True)
         ) if tcol in m.columns else None
    va = (m.dropna(subset=[vcol])[['step', vcol]]
            .rename(columns={vcol: 'val_loss'}).sort_values('step').reset_index(drop=True)
         ) if vcol in m.columns else None
    return tr, va


def _ft_metrics(size, quality):
    q_dir = FT_ROOT / str(size) / str(quality)
    # Fine-tune script copies metrics.csv into loss/ via State._save_loss_curves.
    m = q_dir / 'loss' / 'metrics.csv'
    if m.exists():
        return _read_metrics(m)
    for cand in sorted(q_dir.glob('checkpoints/**/version_*/metrics.csv')):
        return _read_metrics(cand)
    return None, None


def _sc_metrics(size, quality):
    base = SC_ROOT / str(size) / str(quality) / 'results' / 'State' / 'model'
    m = base / 'loss' / 'metrics.csv'
    if m.exists():
        return _read_metrics(m)
    for cand in sorted(base.glob('checkpoints/**/version_*/metrics.csv')):
        return _read_metrics(cand)
    return None, None


def _mi(mi_root: Path):
    """Read MI/<seed>/Y_<signal>_<quality>/lmi_mutual_information.txt -> {signal: value}."""
    out = {}
    if not mi_root.is_dir():
        return out
    for sig_dir in mi_root.iterdir():
        if not sig_dir.is_dir():
            continue
        m = SIGNAL_RE.match(sig_dir.name)
        signal = m.group(1) if m else sig_dir.name
        f = sig_dir / 'lmi_mutual_information.txt'
        if f.exists():
            try:
                out[signal] = float(f.read_text().strip())
            except ValueError:
                pass
    return out


rows = []
CURVES_FT   = {}   # (size, quality) -> (train_df, val_df)
CURVES_SC   = {}

sizes = sorted({int(p.name) for p in FT_ROOT.iterdir() if p.is_dir() and p.name.isdigit()}) if FT_ROOT.is_dir() else []
if not sizes:
    sizes = [100, 215, 464, 1000, 2154, 4641, 10000, 21544, 46415, 100000]
qualities_all = [0.0012346, 0.0025982, 0.0054682, 0.0115083, 0.02422,
                 0.050973, 0.1072766, 0.225772, 0.4751547, 1.0]

for size in sizes:
    for quality in qualities_all:
        ft_tr, ft_va = _ft_metrics(size, quality)
        sc_tr, sc_va = _sc_metrics(size, quality)
        CURVES_FT[(size, quality)] = (ft_tr, ft_va)
        CURVES_SC[(size, quality)] = (sc_tr, sc_va)

        ft_mi = _mi(FT_ROOT / str(size) / str(quality) / 'MI' / str(MI_SEED))
        sc_mi = _mi(SC_ROOT / str(size) / str(quality) / 'results' / 'State' / 'model' / 'MI' / str(MI_SEED))
        row = {'size': size, 'quality': quality}
        for s, v in ft_mi.items():
            row[f'ft_mi_{s}'] = v
        for s, v in sc_mi.items():
            row[f'sc_mi_{s}'] = v
        rows.append(row)

df = pd.DataFrame(rows).sort_values(['size', 'quality']).reset_index(drop=True)
mi_signals = sorted({c.split('_', 2)[2] for c in df.columns if c.startswith(('ft_mi_', 'sc_mi_'))})
print(f'{len(df)} cells   signals={mi_signals}   sizes={sizes}')
df.head()

In [ ]:
# Which (size, quality) cells have fine-tune curves / MI on disk.
ft_tr_have = {k for k, (t, v) in CURVES_FT.items() if t is not None and len(t)}
ft_va_have = {k for k, (t, v) in CURVES_FT.items() if v is not None and len(v)}
sc_tr_have = {k for k, (t, v) in CURVES_SC.items() if t is not None and len(t)}
sc_va_have = {k for k, (t, v) in CURVES_SC.items() if v is not None and len(v)}
print(f'fine-tune   : train={len(ft_tr_have)}/{len(CURVES_FT)}   val={len(ft_va_have)}/{len(CURVES_FT)}')
print(f'from-scratch: train={len(sc_tr_have)}/{len(CURVES_SC)}   val={len(sc_va_have)}/{len(CURVES_SC)}')

## Train / validation loss — fine-tune vs from-scratch

One row per **quality**, two columns (train, val). **Both axes are pinned to shared limits** (global min/max across both training regimes) so the fine-tune's small-step curve and the 15k-step scratch curve can be visually compared without per-panel rescaling.

Fine-tune is the largest available dataset size so the curves converge; from-scratch is matched to the same size.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PLOT_SIZE = max(sizes)   # fine-tune at the largest size has the cleanest curves to compare against scratch
TRAIN_SMOOTH_WINDOW = 200
SKIP_STEPS = 20

def _smooth(tr):
    if tr is None or not len(tr):
        return tr
    tr = tr[tr['step'] >= SKIP_STEPS].copy()
    if not len(tr):
        return tr
    tr['train_loss'] = tr['train_loss'].rolling(
        window=TRAIN_SMOOTH_WINDOW, min_periods=10, center=True).mean()
    return tr.dropna(subset=['train_loss'])

# Collect every (train, val) series we will actually plot so we can compute shared axis limits.
all_train, all_val = [], []
for quality in qualities_all:
    for store in (CURVES_FT, CURVES_SC):
        tr, va = store.get((PLOT_SIZE, quality), (None, None))
        tr = _smooth(tr)
        if tr is not None and len(tr):
            all_train.append(tr)
        if va is not None and len(va):
            va_clip = va[va['step'] >= SKIP_STEPS]
            if len(va_clip):
                all_val.append(va_clip)

def _pad(lo, hi, frac=0.05):
    lo = float(lo); hi = float(hi); pad = (hi - lo) * frac
    return lo - pad, hi + pad

if all_train:
    train_lo, train_hi = _pad(min(t['train_loss'].min() for t in all_train),
                              max(t['train_loss'].max() for t in all_train))
else:
    train_lo, train_hi = 0.0, 1.0
if all_val:
    val_lo, val_hi = _pad(min(v['val_loss'].min() for v in all_val),
                          max(v['val_loss'].max() for v in all_val))
else:
    val_lo, val_hi = train_lo, train_hi
max_step = max(
    [int(t['step'].iloc[-1]) for t in all_train] + [int(v['step'].iloc[-1]) for v in all_val] or [1],
)

n_rows = len(qualities_all)
fig, axes = plt.subplots(n_rows, 2, figsize=(11, 2.4 * n_rows), squeeze=False)

FT_COLOR, SC_COLOR = 'tab:orange', 'tab:blue'
for r, quality in enumerate(qualities_all):
    ax_tr, ax_va = axes[r, 0], axes[r, 1]
    for store, color, label in [(CURVES_FT, FT_COLOR, 'fine-tune SE-100M'),
                                (CURVES_SC, SC_COLOR, 'from-scratch State')]:
        tr, va = store.get((PLOT_SIZE, quality), (None, None))
        tr_s = _smooth(tr)
        if tr_s is not None and len(tr_s):
            ax_tr.plot(tr_s['step'].values, tr_s['train_loss'].values,
                       color=color, linewidth=1.3, alpha=0.9)
        if va is not None and len(va):
            va_c = va[va['step'] >= SKIP_STEPS]
            if len(va_c):
                ax_va.plot(va_c['step'].values, va_c['val_loss'].values,
                           color=color, linewidth=1.3, marker='o', markersize=4)
    ax_tr.set_ylabel(f'q={quality:.4g}\nloss')
    # Pin axes identically across every row so curves are directly comparable.
    for ax, lo, hi in [(ax_tr, train_lo, train_hi), (ax_va, val_lo, val_hi)]:
        ax.set_xlim(0, max_step)
        ax.set_ylim(lo, hi)
        ax.grid(alpha=0.3)

axes[0, 0].set_title(f'train_loss (rolling mean, window={TRAIN_SMOOTH_WINDOW} steps)')
axes[0, 1].set_title('val_loss (per-epoch, raw)')
axes[-1, 0].set_xlabel('step')
axes[-1, 1].set_xlabel('step')

handles = [
    Line2D([0], [0], color=FT_COLOR, linewidth=2, label='fine-tune SE-100M (1 epoch, lr=1e-5)'),
    Line2D([0], [0], color=SC_COLOR, linewidth=2, label='from-scratch State (15k steps)'),
]
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=9, title='model')
fig.suptitle(f'Loss curves — fine-tune SE-100M vs from-scratch State (PBMC size={PLOT_SIZE}, shared y-axis)', y=1.0)
plt.tight_layout()
plt.show()

## MI comparison — side-by-side, shared X and Y axes

For each MI signal we plot two panels: from-scratch on the left, fine-tune on the right. Both panels share the same X (quality, log) and Y (MI, nats) limits so the curves are directly comparable at a glance. One line per dataset size; the same color encodes the same size across the left/right panels.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.cm import viridis
from matplotlib.lines import Line2D

size_color = {s: viridis(i / max(1, len(sizes) - 1)) for i, s in enumerate(sizes)}

for signal in mi_signals:
    ft_col, sc_col = f'ft_mi_{signal}', f'sc_mi_{signal}'
    if ft_col not in df.columns and sc_col not in df.columns:
        continue

    # Shared axis limits across BOTH panels — computed once per signal.
    y_vals = pd.concat([df[ft_col].dropna() if ft_col in df.columns else pd.Series(dtype=float),
                        df[sc_col].dropna() if sc_col in df.columns else pd.Series(dtype=float)])
    if y_vals.empty:
        print(f'  signal={signal!r}: no MI values on disk yet — skipping.')
        continue
    y_pad = (y_vals.max() - y_vals.min()) * 0.05 or 0.01
    y_lim = (float(y_vals.min()) - y_pad, float(y_vals.max()) + y_pad)
    x_lim = (min(qualities_all) * 0.7, max(qualities_all) * 1.3)

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
    for ax, col, title in [(ax_l, sc_col, 'from-scratch State'),
                           (ax_r, ft_col, 'fine-tune SE-100M')]:
        if col not in df.columns:
            ax.set_title(f'{title} — no MI column')
            continue
        for size in sizes:
            sub = df[(df['size'] == size) & df[col].notna()].sort_values('quality')
            if sub.empty:
                continue
            ax.plot(sub['quality'].values, sub[col].values,
                    marker='o', color=size_color[size], linewidth=1.3, label=f'N={size}')
        ax.set_xscale('log')
        ax.set_title(title)
        ax.set_xlabel('quality (downsampling factor)')
        ax.grid(alpha=0.3)
        ax.set_xlim(*x_lim)
        ax.set_ylim(*y_lim)
    ax_l.set_ylabel(f'MI({signal})  [nats]')

    handles = [Line2D([0], [0], color=size_color[s], marker='o', label=f'N={s}') for s in sizes]
    fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8, title='dataset size')
    fig.suptitle(f'MI({signal}) — from-scratch vs fine-tune SE-100M on PBMC (shared axes)', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Delta table: fine-tune minus from-scratch, averaged across qualities per (size, signal).
# Positive = fine-tune helps. Handy sanity check for whether SE-100M is adding signal.
delta_rows = []
for signal in mi_signals:
    ft_col, sc_col = f'ft_mi_{signal}', f'sc_mi_{signal}'
    if ft_col not in df.columns or sc_col not in df.columns:
        continue
    tmp = df[['size', 'quality', ft_col, sc_col]].dropna()
    tmp = tmp.assign(delta=tmp[ft_col] - tmp[sc_col])
    per_size = tmp.groupby('size')['delta'].agg(['mean', 'std', 'count'])
    per_size.columns = [f'{signal}_{c}' for c in per_size.columns]
    delta_rows.append(per_size)
if delta_rows:
    delta_df = pd.concat(delta_rows, axis=1)
    print('Mean (fine-tune - from-scratch) MI per size, averaged across qualities:')
    print(delta_df.round(4).to_string())
else:
    print('No overlapping (ft, sc) MI values available yet.')